In [14]:
from typing import List, Dict
import ujson as json
from pathlib import Path
from openai import OpenAI
from datetime import datetime

# Self-ask / self-instruct
# Self-Ask + filter
# DeepEval Synthetic Module
# Langchain SyntheticQAEvaluator

# 1) Helper functions for Dataset Generation

## Generating slices from chunks

In [15]:
# turning chunks to slices

CHUNK_DIR = Path("data/rag_chunks_v2")

def load_chunks(path: Path) -> List[Dict]:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)  # your file seems to be a JSON list



def make_slices_from_chunks(chunks: List[Dict],
                            max_chars: int = 5000,
                            min_chars: int = 2000,
                        ) -> List[Dict]:

    slices = []
    cur_text = []
    cur_len = 0
    cur_chunk_ids = []

    # assume chunks are already in reading order
    for ch in chunks:
        if ch.get("type") != "paragraph":
            continue
        t = (ch.get("content") or "").strip()
        if not t:
            continue

        if cur_len + len(t) > max_chars and cur_len >= min_chars:
            
            # close current slice
            slice_idx = len(slices)
            doc_id = ch["id"].split("_")[0]   # adjust if you store doc_id elsewhere
            
            slices.append({
                "doc_id": doc_id,
                "slice_id": f"{doc_id}_slice_{slice_idx}",
                "text": "\n\n".join(cur_text),
                "chunk_ids": cur_chunk_ids,
            })
            cur_text, cur_len, cur_chunk_ids = [], 0, []

        # add to current slice
        cur_text.append(t)
        cur_len += len(t)
        cur_chunk_ids.append(ch["id"])

    # flush last slice
    if cur_text:
        doc_id = chunks[0]["id"].split("_")[0]
        slice_idx = len(slices)
        
        slices.append({
            "doc_id": doc_id,
            "slice_id": f"{doc_id}_slice_{slice_idx}",
            "text": "\n\n".join(cur_text),
            "chunk_ids": cur_chunk_ids,
        })

    return slices

In [35]:
# test slices

path = CHUNK_DIR / "2211.17192.rag.chunks.json"
chunks = load_chunks(path)
slices = make_slices_from_chunks(chunks)

print(len(slices), "slices")
print(slices[0]["text"][:2000]) 

9 slices
Given the importance of large autoregressive models and specifically large Transformers, several approaches were

Speculative execution (Burton, 1985; Hennessy & Patterson, 2012) is an optimization technique, common in processors, where a task is performed in parallel to verifying if it's actually needed - the payoff being increased concurrency. A well-known example of speculative execution is branch prediction. For speculative execution to be effective, we need an efficient mechanism to suggest tasks to execute that are likely to be needed. In this work, we generalize speculative execution to the stochastic setting - where a task might be needed with some probability. Applying this to decoding from autoregressive models like Transformers, we sample generations from more efficient approximation models as speculative prefixes for the slower target models . With a novel sampling method, speculative sampling , we maximize the probability of these speculative tasks to

[START] jap

## LLM Call

In [36]:
# API Calls to ChatGPT 

from dotenv import load_dotenv

load_dotenv("keys.env") 
client = OpenAI()
MODEL = "gpt-4.1-mini"  


def get_response(prompt: str) -> str:
    resp = client.responses.create(
        model=MODEL,
        input=[{"role": "user",
                "content": [{"type": "input_text", "text": prompt}]}],
    )
    return resp.output_text.strip()


In [37]:
# --- helper to safely extract JSON ---
def extract_json(raw: str) -> dict:
    raw = raw.strip()

    # Strip ``` fences if present
    if raw.startswith("```"):
        lines = raw.splitlines()
        lines = [ln for ln in lines if not ln.strip().startswith("```")]
        raw = "\n".join(lines).strip()

    # First try: as-is
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        pass

    # Second try: escape backslashes (for \alpha, \nabla, etc.)
    try:
        fixed = raw.replace("\\", "\\\\")
        return json.loads(fixed)
        
    except json.JSONDecodeError as e:
        print("JSON parse error even after fixing backslashes:", e)
        print("Raw output:\n", raw)
        # Fallback: return empty structure so rest of pipeline doesn't crash
        return {"qas": []}

# 2) Dataset Generation with Self-Instruct

In [38]:
SELF_INSTRUCT_PROMPT = """
You are a helpful assistant reading a research paper excerpt.

TEXT:
\"\"\"{text}\"\"\"

Generate {n_qas} diverse question-answer pairs that can be answered *directly and unambiguously* from this text alone.

Requirements:
- Cover different types: definitions, methods, motivations, comparisons, results.
- Make questions specific, not vague.
- Answers should be concise and copy or paraphrase the text.
- Return JSON as a list under key "qas", like:
  {{"qas": [{{"question": "...", "answer": "..."}}, ...]}}
"""

In [39]:
QA = Dict[str, str]

def generate_qas_self_instruct(doc_id: str, text: str, n_qas: int = 5) -> List[QA]:
    
    prompt = SELF_INSTRUCT_PROMPT.format(text=text, n_qas=n_qas)
    response = get_response(prompt)
    data = extract_json(response)

    qas: List[QA] = []
    for qa in data.get("qas", []):
        qas.append({
            "question": qa["question"],
            "answer": qa["answer"],
            "doc_id": doc_id,
            "source_text": text,
        })
    return qas

### sample QA generation

In [40]:
slice0 = slices[0]
qas = generate_qas_self_instruct(
    doc_id=slice0["doc_id"],
    text=slice0["text"],
    n_qas=5,
)

for qa in qas:
    print("Q:", qa["question"])
    print("A:", qa["answer"])
    print("----")

Q: What is speculative execution as described in the text?
A: Speculative execution is an optimization technique where a task is performed in parallel to verifying if it's actually needed, increasing concurrency.
----
Q: What novel method do the authors introduce to generalize speculative execution in their work?
A: The authors introduce speculative sampling, a novel sampling method to generalize speculative execution to the stochastic setting.
----
Q: How does the proposed speculative decoding accelerate autoregressive model decoding?
A: It uses a more efficient approximation model to generate multiple speculative completions in parallel, then evaluates these guesses with the target model, accepting those that preserve the distribution, thus reducing the number of serial runs needed.
----
Q: What are the main contributions of the paper?
A: 1) Generalization of speculative execution to the stochastic setting with speculative sampling, and 2) Speculative decoding mechanism that accelera

### Building and saving dataset from all chunks

In [11]:
# create folder: data/QnA/
save_dir = Path("data") / "QnA"
save_dir.mkdir(parents=True, exist_ok=True)

OUT_PATH = save_dir / "eval_qas_self_instruct.jsonl"
DATASET_NAME = "arxiv_scale_rag_self_instruct_v1" 

In [16]:


def build_eval_qas_jsonl(
    chunk_dir: Path = CHUNK_DIR,
    out_path: Path = OUT_PATH,
    n_qas_per_slice: int = 5,
    max_docs: int | None = None,   # for testing; set None for all
):
    with out_path.open("w", encoding="utf-8") as out_f:
        for i, path in enumerate(sorted(chunk_dir.glob("*.rag.chunks.json"))):
            if max_docs is not None and i >= max_docs:
                break

            chunks = load_chunks(path)
            slices = make_slices_from_chunks(chunks, max_chars=3000, min_chars=500)

            for slice_idx, s in enumerate(slices):
                qas = generate_qas_self_instruct(
                    doc_id=s["doc_id"],
                    text=s["text"],
                    n_qas=n_qas_per_slice,
                )

                for qa_idx, qa in enumerate(qas):
                    record = {
                        # identifiers
                        "id": f"{s['doc_id']}_slice{slice_idx}_qa{qa_idx}",
                        "dataset": DATASET_NAME,
                        "doc_id": s["doc_id"],
                        "slice_id": s["slice_id"],
                        "slice_index": slice_idx,
                        "chunk_ids": s["chunk_ids"],

                        # QA
                        "question": qa["question"],
                        "answer": qa["answer"],

                        # optional, but useful for debugging / later analysis
                        "source_text": s["text"],  # or s["text"][:2000] if size is an issue

                        # meta
                        "generator_model": MODEL,  # your OpenAI model name, e.g. "gpt-4.1-mini"
                        "created_at": datetime.utcnow().isoformat(),
                    }
                    out_f.write(json.dumps(record) + "\n")

            print(f"Processed {path.name}: {len(slices)} slices")


In [28]:
# run a small test first
build_eval_qas_jsonl(max_docs=None)  # try on 3 papers first

Processed 2001.08361.rag.chunks.json: 25 slices
Processed 2005.03141.rag.chunks.json: 9 slices
Processed 2101.03961.rag.chunks.json: 31 slices
Processed 2106.06967.rag.chunks.json: 15 slices
Processed 2203.15556.rag.chunks.json: 24 slices
Processed 2211.10438.rag.chunks.json: 18 slices
Processed 2211.17192.rag.chunks.json: 16 slices
Processed 2303.11312.rag.chunks.json: 26 slices
Processed 2303.11313.rag.chunks.json: 21 slices
Processed 2306.00978.rag.chunks.json: 21 slices
Processed 2306.10209.rag.chunks.json: 24 slices
Processed 2306.14048.rag.chunks.json: 48 slices
Processed 2307.08691.rag.chunks.json: 12 slices
Processed 2309.06180.rag.chunks.json: 30 slices
Processed 2310.01801.rag.chunks.json: 17 slices
Processed 2312.00752.rag.chunks.json: 50 slices
Processed 2312.11514.rag.chunks.json: 28 slices
Processed 2401.10774.rag.chunks.json: 26 slices
Processed 2401.18059.rag.chunks.json: 25 slices
Processed 2406.03243.rag.chunks.json: 34 slices
Processed 2408.00724.rag.chunks.json: 21 

# 2) Evaluating Scalability

In [ ]:
!kill -9 -1

In [ ]:
# and save the results
save_log(run_log, Path("run_log.graph_rag_v1.jsonl"))